In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q google-genai gspread
import json
import time
import os
from google import genai
from google.genai import types
from tqdm import tqdm

In [3]:
from google.colab import auth
import gspread
from google.auth import default

In [ ]:
# 1. XÁC THỰC TÀI KHOẢN GOOGLE SHEETS
print("Đang xác thực tài khoản Google...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 2. KHỞI TẠO CLIENT GENAI
# Lưu ý: Tuyệt đối không chia sẻ API Key cho người khác
API_KEY = ""
client = genai.Client(api_key=API_KEY)
system_instruction = "Bạn là chuyên gia phân tích tài liệu y học học thuật."

# 3. KẾT NỐI VÀO GOOGLE SHEETS BẰNG ID VÀ GID
sheet_id = '1K_C3MZ7oeSQiOWsXwc1zHaTBPPZBb6HD8tRbody_zfk'
gid_target = 1073799587

print("Đang mở Google Sheets...")
spreadsheet = gc.open_by_key(sheet_id)

# Tìm đúng worksheet dựa trên gid
worksheet = None
for ws in spreadsheet.worksheets():
    if ws.id == gid_target:
        worksheet = ws
        break

if not worksheet:
    raise ValueError(f"Không tìm thấy Sheet nào có gid={gid_target} trong file này.")

Đang xác thực tài khoản Google...
Đang mở Google Sheets...


In [ ]:
# 4. ĐỌC DỮ LIỆU & TÌM VỊ TRÍ CỘT
all_values = worksheet.get_all_values()
headers = all_values[0] # Dòng đầu tiên là tiêu đề

# Lấy số thứ tự (index 1-based) của các cột cần thiết để cập nhật chính xác
try:
    col_noidung_idx = headers.index('noi_dung_bai_bao') + 1
    col_lv1_idx = headers.index('lv1') + 1
    col_lv2_idx = headers.index('lv2') + 1
    col_lv3_idx = headers.index('lv3') + 1
except ValueError as e:
    raise ValueError("Lỗi: Không tìm thấy các cột 'noi_dung_bai_bao', 'lv1', 'lv2' hoặc 'lv3' ở dòng tiêu đề.")

# 5. XỬ LÝ TỪNG DÒNG (Bỏ qua dòng 1 là Header)
for i in tqdm(range(1, len(all_values)), desc="Đang sinh dữ liệu"):
    row_data = all_values[i]
    sheet_row = i + 1 # Index dòng thực tế trên Sheet (1-based)

    # Nếu file Sheet có các dòng trống ở cuối thì bỏ qua
    if len(row_data) < col_noidung_idx:
        continue

    noi_dung = str(row_data[col_noidung_idx - 1]).strip()

    # Kiểm tra xem dòng này đã được tóm tắt chưa (nếu lv1 đã có chữ thì skip để đỡ tốn API)
    # Nếu muốn chạy lại toàn bộ, bạn có thể xóa/comment khối if này.
    lv1_current = row_data[col_lv1_idx - 1].strip() if len(row_data) >= col_lv1_idx else ""
    if lv1_current != "":
        continue

    if not noi_dung or len(noi_dung) < 100 or noi_dung.lower() == 'nan':
        continue

    # Giới hạn số lượng ký tự đầu vào
    text_input = noi_dung[:150000]
    user_prompt = f"""Nhiệm vụ:
Đọc nội dung tài liệu/bài báo cáo y học được cung cấp và tạo bản tóm tắt theo 3 cấp độ khác nhau.

YÊU CẦU CHUNG:
- Không bịa thêm dữ liệu ngoài tài liệu. Nếu tài liệu không có thông tin nào thì ghi: “Không đề cập”.
- Giữ nguyên các thuật ngữ y khoa quan trọng. Nếu có số liệu nghiên cứu, thuốc, chỉ số xét nghiệm → giữ nguyên đơn vị, không làm tròn tùy tiện.
- Nếu có guideline hoặc trial name → giữ nguyên tên tiếng Anh.
- Văn phong mạch lạc, logic. Ưu tiên nội dung lâm sàng, cơ chế bệnh sinh, chẩn đoán, điều trị và kết luận.

========================
LEVEL 1 — SƠ LƯỢC
Mục tiêu: Tạo bản tóm tắt ngắn gọn
- Chỉ dùng các gạch đầu dòng ngắn. Mỗi ý tối đa 1–2 dòng.
- Tập trung: Chủ đề chính, Cơ chế bệnh/chẩn đoán/điều trị, Kết luận quan trọng.
- Không diễn giải dài, Không phân tích sâu.

========================
LEVEL 2 — DỄ HIỂU
Mục tiêu: Giải thích tài liệu bằng ngôn ngữ đời thường để người không chuyên vẫn hiểu.
- Văn phong diễn giải súc tích rõ ràng, ngôn ngữ đời thường, không so sánh/ẩn dụ. Giải thích thuật ngữ ở mức độ đơn giản.
- Giải thích: Cơ chế bệnh sinh, Tác dụng thuốc, Ý nghĩa xét nghiệm, Tiến triển bệnh theo cách dễ hiểu.
- Không được làm sai bản chất y học hoặc bỏ mất ý quan trọng.
- Độ dài: 4–10 đoạn tùy độ phức tạp tài liệu.

========================
LEVEL 3 — CHUYÊN SÂU
Mục tiêu: Viết bản tóm tắt thành các đoạn văn học thuật hoàn chỉnh cho người có chuyên môn y khoa.
- Giữ nguyên: Thuật ngữ Latin, Tên thuốc, Chỉ số xét nghiệm, Tên guideline/trial.
- Phân tích logic theo cấu trúc: Tổng quan -> Cơ chế bệnh sinh -> Triệu chứng/lâm sàng -> Chẩn đoán -> Điều trị -> Kết quả nghiên cứu -> Kết luận chuyên môn.
- Văn phong học thuật, chặt chẽ. Không đơn giản hóa thuật ngữ chuyên môn.

========================
TRẢ VỀ DUY NHẤT ĐỊNH DẠNG JSON SAU (Không kèm giải thích gì thêm):
{{
  "chu_de": "<Trích xuất Chủ đề chính của tài liệu>",
  "lv1": "<Nội dung LEVEL 1>",
  "lv2": "<Nội dung LEVEL 2>",
  "lv3": "<Nội dung LEVEL 3>"
}}

========================
TÀI LIỆU GỐC:
{text_input}"""

    max_retries = 4

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=0.2,
                    response_mime_type="application/json",
                )
            )

            result = json.loads(response.text)

            lv1_text = result.get("lv1", "")
            lv2_text = result.get("lv2", "")
            lv3_text = result.get("lv3", "")

            # CẬP NHẬT TRỰC TIẾP LÊN GOOGLE SHEETS
            # Dùng update_cells để đẩy cả 3 ô (lv1, lv2, lv3) lên cùng 1 lúc (tiết kiệm request API)
            cells_to_update = [
                gspread.Cell(row=sheet_row, col=col_lv1_idx, value=lv1_text),
                gspread.Cell(row=sheet_row, col=col_lv2_idx, value=lv2_text),
                gspread.Cell(row=sheet_row, col=col_lv3_idx, value=lv3_text)
            ]
            worksheet.update_cells(cells_to_update)

            # Đợi một chút tránh bị giới hạn API từ cả Google Sheets và GenAI
            time.sleep(10)
            break

        except json.JSONDecodeError:
            print(f"\n[Lỗi JSON] Mô hình sinh sai cấu trúc ở dòng {sheet_row}. Đang thử sinh lại... (Lần {attempt + 1}/{max_retries})")
            time.sleep(5)

        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg or "Quota" in error_msg:
                print(f"\n[Quá tải API] Đợi 60 giây để khôi phục... (Lần {attempt + 1}/{max_retries})")
                time.sleep(60)
            else:
                print(f"\n[Lỗi] Bài báo dòng {sheet_row} gặp lỗi: {e}")
                time.sleep(5)

print("\n🎉 Hoàn tất! Toàn bộ dữ liệu đã được điền trực tiếp vào Google Sheets.")

Đang sinh dữ liệu:   6%|▌         | 11/179 [02:17<50:25, 18.01s/it]